# Day 6 - CNN Image Classification (RetailMind AI)

**Goal:** build a CNN that automatically sorts product images into catalog categories.

Steps: prototype on Fashion-MNIST, then train on real product photos, then check which categories get confused.

## 1. Setup

In [1]:
import sys
sys.path.append("..")

import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

from src.config import ROOT
from src.cnn import build_cnn

keras.utils.set_random_seed(42)
MODEL_DIR = ROOT / "models"

## 2. Prototype on Fashion-MNIST
Small 28x28 grayscale clothing pictures in 10 classes. It trains fast, so it is a quick way to check the CNN design before using real photos.

In [2]:
(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr = X_tr[..., np.newaxis]     # add a channel axis: (28, 28) -> (28, 28, 1)
X_te = X_te[..., np.newaxis]

proto = build_cnn(input_shape=(28, 28, 1), num_classes=10)
proto.fit(X_tr, y_tr, epochs=3, batch_size=128, validation_split=0.1, verbose=2)

loss, acc = proto.evaluate(X_te, y_te, verbose=0)
print("Fashion-MNIST test accuracy:", round(acc, 4))

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 4us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/3
422/422 - 18s - 42ms/step - accuracy: 0.7583 - loss: 0.6713 - val_accuracy: 0.8545 - val_loss: 0.3931
Epoch 2/3
422/422 - 14s - 32ms/step - accuracy: 0.8430 - loss: 0.4332 - val_accuracy: 0.8732 - val_loss: 0.3360
Epoch 3/3
422/422 - 14s - 32ms/step - accuracy: 0.8630 - loss: 0.3791 - val_accuracy: 0.8867 - val_loss: 0.3027
Fashion-MNIST test accuracy: 0.8819


The architecture works, so we can move on to the real data.

## 3. Load the Fashion Product Images
Dataset: Kaggle `paramaggarwal/fashion-product-images-small` (a `styles.csv` file with the labels and an `images` folder), stored in `data/fashion-product-images-small/`.

In [3]:
DATA_DIR = ROOT / "data" / "fashion-product-images-small"

if not (DATA_DIR / "styles.csv").exists():
    raise FileNotFoundError("Dataset not found at " + str(DATA_DIR))

styles = pd.read_csv(DATA_DIR / "styles.csv", on_bad_lines="skip")   # a few rows in the file are broken
print("Rows in styles.csv:", len(styles))
styles["articleType"].value_counts().head(12)

Rows in styles.csv: 44424


articleType
Tshirts         7067
Shirts          3217
Casual Shoes    2845
Watches         2542
Sports Shoes    2036
Kurtas          1844
Tops            1762
Handbags        1759
Heels           1323
Sunglasses      1073
Wallets          936
Flip Flops       914
Name: count, dtype: int64

To keep training quick, we use the **10 biggest categories** and at most **600 images each**.

In [4]:
top_types = styles["articleType"].value_counts().head(10).index
styles = styles[styles["articleType"].isin(top_types)]
styles = styles.sample(frac=1, random_state=42).groupby("articleType").head(600)
print("Images to load:", len(styles))

Images to load: 6000


Load each image at 80x60 pixels (the dataset's small size) and keep its category.

In [5]:
images, labels = [], []
for _, row in styles.iterrows():
    path = DATA_DIR / "images" / (str(row["id"]) + ".jpg")
    if not path.exists():
        continue
    img = keras.utils.load_img(path, target_size=(80, 60))   # (height, width)
    images.append(keras.utils.img_to_array(img))
    labels.append(row["articleType"])

X = np.array(images, dtype="uint8")
class_names = sorted(set(labels))
y = np.array([class_names.index(name) for name in labels])
print("Loaded:", X.shape, "| classes:", class_names)

Loaded: (6000, 80, 60, 3) | classes: ['Casual Shoes', 'Handbags', 'Heels', 'Kurtas', 'Shirts', 'Sports Shoes', 'Sunglasses', 'Tops', 'Tshirts', 'Watches']


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train:", len(X_train), "| Test:", len(X_test))

Train: 4800 | Test: 1200


## 4. Build the CNN with augmentation
The CNN is `conv, pool, conv, pool, dropout, dense` (see `src/cnn.py`). With `augment=True` it also randomly flips, rotates and zooms the training images, so it sees more variety and overfits less. This only happens while training.

In [7]:
cnn = build_cnn(input_shape=(80, 60, 3), num_classes=len(class_names), augment=True)
cnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Layer (type)             ┃ Output Shape      ┃   Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ random_flip (RandomFlip) │ (None, 80, 60, 3) │         0 │
├──────────────────────────┼───────────────────┼───────────┤
│ random_rotation          │ (None, 80, 60, 3) │         0 │
│ (RandomRotation)         │                   │           │
├──────────────────────────┼───────────────────┼───────────┤
│ random_zoom (RandomZoom) │ (None, 80, 60, 3) │         0 │
├──────────────────────────┼───────────────────┼───────────┤
│ rescaling_1 (Rescaling)  │ (None, 80, 60, 3) │         0 │
├──────────────────────────┼───────────────────┼───────────┤
│ conv2d_2 (Conv2D)        │ (None, 78, 58,    │       896 │
│                          │ 32)               │           │
├──────────────────────────┼───────────────────┼───────────┤
│ max_pooling2d_2          │ (None, 39, 29,    │         0 │
│ (MaxPooling2D)           │ 32)               │           │
├──────────────────────────┼───────────────────┼───────────┤
│ conv2d_3 (Conv2D)        │ (None, 37, 27,    │    18,496 │
│                          │ 64)               │           │
├──────────────────────────┼───────────────────┼───────────┤
│ max_pooling2d_3          │ (None, 18, 13,    │         0 │
│ (MaxPooling2D)           │ 64)               │           │
├──────────────────────────┼───────────────────┼───────────┤
│ dropout_2 (Dropout)      │ (None, 18, 13,    │         0 │
│                          │ 64)               │           │
├──────────────────────────┼───────────────────┼───────────┤
│ flatten_1 (Flatten)      │ (None, 14976)     │         0 │
├──────────────────────────┼───────────────────┼───────────┤
│ dense_2 (Dense)          │ (None, 128)       │ 1,917,056 │
├──────────────────────────┼───────────────────┼───────────┤
│ dropout_3 (Dropout)      │ (None, 128)       │         0 │
├──────────────────────────┼───────────────────┼───────────┤
│ dense_3 (Dense)          │ (None, 10)        │     1,290 │
└──────────────────────────┴───────────────────┴───────────┘

 Total params: 1,937,738 (7.39 MB)

 Trainable params: 1,937,738 (7.39 MB)

 Non-trainable params: 0 (0.00 B)

## 5. Train

In [8]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

history = cnn.fit(
    X_train, y_train,
    validation_split=0.2, epochs=20, batch_size=64,
    callbacks=[early_stop], verbose=2,
)

Epoch 1/20
60/60 - 12s - 202ms/step - accuracy: 0.3729 - loss: 1.7150 - val_accuracy: 0.7010 - val_loss: 0.7807
Epoch 2/20
60/60 - 7s - 124ms/step - accuracy: 0.6224 - loss: 1.0128 - val_accuracy: 0.7312 - val_loss: 0.6695
Epoch 3/20
60/60 - 7s - 120ms/step - accuracy: 0.6896 - loss: 0.8566 - val_accuracy: 0.7823 - val_loss: 0.5652
Epoch 4/20
60/60 - 7s - 123ms/step - accuracy: 0.7151 - loss: 0.7579 - val_accuracy: 0.7802 - val_loss: 0.5462
Epoch 5/20
60/60 - 7s - 123ms/step - accuracy: 0.7474 - loss: 0.7002 - val_accuracy: 0.8062 - val_loss: 0.4905
Epoch 6/20
60/60 - 8s - 125ms/step - accuracy: 0.7604 - loss: 0.6511 - val_accuracy: 0.8385 - val_loss: 0.4401
Epoch 7/20
60/60 - 7s - 123ms/step - accuracy: 0.7737 - loss: 0.6135 - val_accuracy: 0.8542 - val_loss: 0.4047
Epoch 8/20
60/60 - 7s - 122ms/step - accuracy: 0.7875 - loss: 0.5927 - val_accuracy: 0.8458 - val_loss: 0.4140
Epoch 9/20
60/60 - 7s - 116ms/step - accuracy: 0.7945 - loss: 0.5729 - val_accuracy: 0.8438 - val_loss: 0.4191


## 6. Evaluate on the test images

In [9]:
probs = cnn.predict(X_test, verbose=0)
pred = probs.argmax(axis=1)
accuracy = round(float((pred == y_test).mean()), 4)
print("Test accuracy:", accuracy)

Test accuracy: 0.8575


### Confusion matrix
Rows are the true category, columns are what the model predicted. Numbers off the diagonal are mistakes.

In [10]:
cm = tf.math.confusion_matrix(y_test, pred).numpy()
pd.DataFrame(cm, index=class_names, columns=class_names)

,Casual Shoes,Handbags,Heels,Kurtas,Shirts,Sports Shoes,Sunglasses,Tops,Tshirts,Watches
Casual Shoes,77,0,14,0,0,29,0,0,0,0
Handbags,1,117,1,0,0,1,0,0,0,0
Heels,7,2,107,0,0,2,1,0,0,1
Kurtas,0,0,0,115,2,0,0,3,0,0
Shirts,0,0,0,2,98,0,0,9,11,0
Sports Shoes,19,1,3,0,0,97,0,0,0,0
Sunglasses,0,0,0,0,0,1,119,0,0,0
Tops,0,3,0,5,10,0,0,86,16,0
Tshirts,0,0,0,0,6,0,0,17,96,1
Watches,0,0,0,0,0,1,1,0,1,117


In [11]:
# the most common mistakes
mistakes = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm[i, j] > 0:
            mistakes.append((cm[i, j], class_names[i], class_names[j]))

mistakes.sort(reverse=True)
for count, true_name, predicted_name in mistakes[:5]:
    print(true_name, "predicted as", predicted_name, ":", count, "times")

Casual Shoes predicted as Sports Shoes : 29 times
Sports Shoes predicted as Casual Shoes : 19 times
Tshirts predicted as Tops : 17 times
Tops predicted as Tshirts : 16 times
Casual Shoes predicted as Heels : 14 times


The list above shows which look-alike categories the model mixes up most. Those are the ones a better model or more data would need to fix next.

## 7. Save the catalog classifier
Saved together with the category names, so a prediction can be turned back into a label.

In [12]:
cnn.save(MODEL_DIR / "catalog_cnn.keras")
with open(MODEL_DIR / "catalog_classes.json", "w") as f:
    json.dump(class_names, f)
print("Saved catalog_cnn.keras and catalog_classes.json")

Saved catalog_cnn.keras and catalog_classes.json
